In [1]:
import os
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Regression Models
from sklearn.linear_model import Lasso
# Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# Setup path
BASE_PATH = 'dataset'
OUTPUT_FOLDER = os.path.join(BASE_PATH, 'output_pipeline')
ARTIFACTS_FOLDER = os.path.join(BASE_PATH, 'artifacts')
MODEL_FOLDER = os.path.join(BASE_PATH, 'model_outputs')
EDA_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')
RESULTS_FOLDER = os.path.join(BASE_PATH, 'results')

os.makedirs(MODEL_FOLDER, exist_ok=True)
os.makedirs(EDA_FOLDER, exist_ok=True)
os.makedirs(RESULTS_FOLDER, exist_ok=True)

In [3]:
# Load fitur (X)
X_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_train.parquet'))
X_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_val.parquet'))
X_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_X_test.parquet'))

In [4]:
# Load target (y) - convert ke Series untuk sklearn
y_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_train.parquet'))
y_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_val.parquet'))
y_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_y_test.parquet'))

In [5]:
# Load metadata ID (untuk traceback hasil prediksi nanti)
ID_train = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_train.parquet'))
ID_val = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_val.parquet'))
ID_test = pd.read_parquet(os.path.join(OUTPUT_FOLDER, 'stage6_ID_test.parquet'))

In [6]:
scaler = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'minmax_scaler.joblib'))
kolom_info = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'kolom_info.joblib'))
winsor = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'winsor_bounds.joblib'))
feat_select = joblib.load(os.path.join(ARTIFACTS_FOLDER, 'feature_selection_info.joblib'))

/home/aliarridha/miniconda3/envs/modeling1/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [7]:
TARGET = kolom_info['target']

In [8]:
y_train_arr = y_train[TARGET].values
y_val_arr = y_val[TARGET].values
y_test_arr = y_test[TARGET].values

In [9]:
print('=== VERIFIKASI LOAD DATA ===')
print(f'X_train : {X_train.shape}')
print(f'X_val : {X_val.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train : {y_train_arr.shape}')
print(f'y_val : {y_val_arr.shape}')
print(f'y_test : {y_test_arr.shape}')
print(f'\nTotal fitur: {X_train.shape[1]}')
print(f'Target : {TARGET}')
print(f'\nNull check:')
print(f' X_train: {X_train.isnull().sum().sum()}')
print(f' X_val : {X_val.isnull().sum().sum()}')
print(f' X_test : {X_test.isnull().sum().sum()}')
print(f'\nRange target:')
print(f' Train: {y_train_arr.min():.2f} - {y_train_arr.max():.2f} Qu/Ha')
print(f' Val : {y_val_arr.min():.2f} - {y_val_arr.max():.2f} Qu/Ha')
print(f' Test : {y_test_arr.min():.2f} - {y_test_arr.max():.2f} Qu/Ha')
print(f'\nArtifak loaded:')
print(f' minmax_scaler : OK')
print(f' kolom_info : {len(kolom_info)} entri')
print(f' winsor_bounds : {len(winsor)} kolom')
print(f'\nFolder output:')
print(f' Model : {MODEL_FOLDER}')
print(f' Gambar : {EDA_FOLDER}')
print('\nData siap. Lanjut ke modeling.')

=== VERIFIKASI LOAD DATA ===
X_train : (2251, 85)
X_val : (450, 85)
X_test : (893, 85)
y_train : (2251,)
y_val : (450,)
y_test : (893,)

Total fitur: 85
Target : produktivitas_kuha

Null check:
 X_train: 0
 X_val : 0
 X_test : 0

Range target:
 Train: 12.32 - 75.54 Qu/Ha
 Val : 18.63 - 71.66 Qu/Ha
 Test : 12.54 - 75.32 Qu/Ha

Artifak loaded:
 minmax_scaler : OK
 kolom_info : 12 entri
 winsor_bounds : 59 kolom

Folder output:
 Model : dataset/model_outputs
 Gambar : dataset/eda_outputs

Data siap. Lanjut ke modeling.


In [10]:
print(ID_test.head())
print(ID_test.columns)

         kabupaten_kota   nama_wilayah_bersih  tahun
0       Kab. Aceh Barat       kab. aceh barat   2024
1       Kab. Aceh Barat       kab. aceh barat   2025
2  Kab. Aceh Barat Daya  kab. aceh barat daya   2024
3  Kab. Aceh Barat Daya  kab. aceh barat daya   2025
4       Kab. Aceh Besar       kab. aceh besar   2024
Index(['kabupaten_kota', 'nama_wilayah_bersih', 'tahun'], dtype='object')


In [11]:
print("Mulai training Lasso Regression")

# inisialisasi model
lasso_model = Lasso(
    alpha=0.01,
    max_iter=10000
)

# training model
lasso_model.fit(X_train, y_train_arr)

print("Training selesai")

Mulai training Lasso Regression
Training selesai


In [12]:
# Prediksi
train_pred = lasso_model.predict(X_train)
val_pred = lasso_model.predict(X_val)
test_pred = lasso_model.predict(X_test)

In [13]:
def hitung_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [14]:
# evaluasi train
train_mae, train_rmse, train_r2 = hitung_metrics(
    y_train_arr,
    train_pred
)

In [15]:
# evaluasi validation
val_mae, val_rmse, val_r2 = hitung_metrics(
    y_val_arr,
    val_pred
)

In [16]:
# evaluasi test
test_mae, test_rmse, test_r2 = hitung_metrics(
    y_test_arr,
    test_pred
)

In [17]:
print("HASIL LASSO REGRESSION")

print("\nTrain")
print(f"MAE  : {train_mae:.4f}")
print(f"RMSE : {train_rmse:.4f}")
print(f"R2   : {train_r2:.4f}")

print("\nValidation")
print(f"MAE  : {val_mae:.4f}")
print(f"RMSE : {val_rmse:.4f}")
print(f"R2   : {val_r2:.4f}")

print("\nTest")
print(f"MAE  : {test_mae:.4f}")
print(f"RMSE : {test_rmse:.4f}")
print(f"R2   : {test_r2:.4f}")

HASIL LASSO REGRESSION

Train
MAE  : 4.6342
RMSE : 5.9533
R2   : 0.6924

Validation
MAE  : 4.6091
RMSE : 5.9524
R2   : 0.6653

Test
MAE  : 4.8909
RMSE : 6.3097
R2   : 0.6428


In [ ]:
# Simpan Model Lasso Regression
joblib.dump(
    lasso_model,
    os.path.join(BASE_PATH, 'models', 'lasso_regression.joblib')
)
print("Model berhasil disimpan.")

In [ ]:
# Simpan metrics
metrics_lasso = pd.DataFrame({
    "model": ["Lasso Regression"],
    "train_mae": [train_mae],
    "train_rmse": [train_rmse],
    "train_r2": [train_r2],
    "val_mae": [val_mae],
    "val_rmse": [val_rmse],
    "val_r2": [val_r2],
    "test_mae": [test_mae],
    "test_rmse": [test_rmse],
    "test_r2": [test_r2]
})

metrics_lasso.to_csv(
    os.path.join(RESULTS_FOLDER, "lasso_regression_metrics.csv"),
    index=False
)
print("Metrics berhasil disimpan.")

In [ ]:
# Simpan hasil prediksi test
hasil_pred_lasso = ID_test.copy()

hasil_pred_lasso["actual"] = y_test_arr
hasil_pred_lasso["prediction_lasso"] = test_pred

hasil_pred_lasso.to_csv(
    os.path.join(RESULTS_FOLDER, "lasso_regression_predictions.csv"),
    index=False
)

print("Hasil prediksi berhasil disimpan.")

# preview hasil
print("\nPreview prediction:")
print(hasil_pred_lasso.head())